# [Kaggle] Closed-question fine-tuning **WITH CoT** — all models, one after another, auto-push to Hugging Face

**Kaggle setup (before running)**
* Upload your dataset zip as a Kaggle Dataset (Kaggle extracts it automatically) and click **+ Add Input** to attach it. `closed_with_CoT.csv` is found automatically under `/kaggle/input`.
* Session options: Accelerator = **GPU T4 x2**, Internet = **On**.
* **Add-ons → Secrets**: add `HF_TOKEN` = your Hugging Face **Write** token and attach it to this notebook.
* Accept the licenses at https://huggingface.co/google/gemma-4-E4B-it and https://huggingface.co/google/medgemma-4b-it.

Cells 1–7 are the same steps as the original notebook. Cells 8–14 are each one original step (load → LoRA → before check → trainer → train → after check → save → push). The last cell runs those steps for every model in `REPO_IDS`, pushing each to the Hub before starting the next.

In [15]:
!pip install -q "unsloth==2026.9.2"

In [16]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # before torch is imported
os.environ["UNSLOTH_RETURN_LOGITS"]   = "1"   # bypass Unsloth fused CE loss (Llama torch.compile crash)
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"   # disable torch.compile patches (same maths, a bit slower)

from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch, gc, shutil

REPO_IDS = {
    "MedGemma-4B": "google/medgemma-4b-it",
    "Qwen2.5-VL-7B": "Qwen/Qwen2.5-VL-7B-Instruct",
    "Llama-11B": "unsloth/Llama-3.2-11B-Vision-Instruct",
    "Qwen3-VL-8B": "Qwen/Qwen3-VL-8B-Instruct",
    "Lingshu-7B": "lingshu-medical-mllm/Lingshu-7B",
    # "Gemma4-E4B": "google/gemma-4-E4B-it",   # not trainable on T4 -> run on Colab L4/A100
}

HF_USERNAME = "Arup330"
DATASET_TAG = "Abdomen"

In [17]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [18]:
# ---------------------------------------------------------------------------
# 3. Load the closed-question CSV from the attached Kaggle dataset
# ---------------------------------------------------------------------------
import os
import glob
import pandas as pd
from PIL import Image

COT_DATA_ROOT = "/kaggle/input/datasets/arups330/slackdataset/Abdomen_Train_with_CoT/CT/train"
csv_paths = [
    os.path.join(COT_DATA_ROOT, "closed_with_CoT.csv"),
]
print(f"Found {len(csv_paths)} CSVs:")
for p in csv_paths:
    print(" ", p)

frames = []
for csv_path in csv_paths:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    frames.append(df)

cot_df = pd.concat(frames, ignore_index=True)

# Keep ONLY closed questions whose answer is Yes / No -- ignore everything else
before = len(cot_df)
ans = cot_df["answer"].astype(str).str.strip().str.lower()
cot_df = cot_df[ans.isin(["yes", "no"])].copy()
cot_df["answer"] = cot_df["answer"].astype(str).str.strip().str.lower().map({"yes": "Yes", "no": "No"})
cot_df = cot_df.reset_index(drop=True)
print(f"Kept {len(cot_df)} / {before} Yes/No rows "
      f"(Yes={(cot_df['answer']=='Yes').sum()}, No={(cot_df['answer']=='No').sum()})")

IMG_COL = "image_file" if "image_file" in cot_df.columns else "img_name"

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    candidates = [
        os.path.join(split_dir, str(img_name)),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, str(img_name).replace("/", "_")),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

print("First image:", resolve_image_path(cot_df.iloc[0]["split_dir"], cot_df.iloc[0][IMG_COL]))

Found 1 CSVs:
  /kaggle/input/datasets/arups330/slackdataset/Abdomen_Train_with_CoT/CT/train/closed_with_CoT.csv
Kept 133 / 150 Yes/No rows (Yes=70, No=63)
First image: /kaggle/input/datasets/arups330/slackdataset/Abdomen_Train_with_CoT/CT/train/xmlab104_source.jpg


In [19]:
# ---------------------------------------------------------------------------
# 4. convert_to_conversation -- same style as Unsloth's radiography example,
#    but: instruction = your closed-question prompt, and the
#    assistant target is the one-word ground-truth answer (Yes / No).
# ---------------------------------------------------------------------------
def systemPrompt(question: str, cot: str) -> str:
    return f"""Context:
- You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.

Objective:
- Answer the user's question by verifying whether it is supported by the visual evidence in the medical image and the provided Step of thinking (CoT).

Inputs:
- Question: {question}
- Step of thinking (CoT): {cot}

Instructions:
1. Examine the medical image carefully.
2. Evaluate your visual findings against the provided Step of thinking.
3. If the Step of thinking is inconsistent with the image, prioritize the image evidence.
4. If the image does not provide sufficient evidence to support a "Yes" answer, return "No."

Output Requirements:
- Output exactly one word.
- Do not provide explanations, punctuation, or additional text.

Valid outputs:
Yes
No
"""

def convert_to_conversation(sample):
    instruction = systemPrompt(sample["question"], sample["CoT"])
    image_path = resolve_image_path(sample["split_dir"], sample[IMG_COL])
    image = Image.open(image_path).convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": image},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}],
        },
    ]
    return {"messages": conversation}
pass

print("Converting rows to Unsloth chat format...")
converted_dataset = [convert_to_conversation(row) for _, row in cot_df.iterrows()]
print(f"Done. {len(converted_dataset)} examples ready.")
print("Example:", converted_dataset[0]["messages"])

Converting rows to Unsloth chat format...
Done. 133 examples ready.
Example: [{'role': 'user', 'content': [{'type': 'text', 'text': 'Context:\n- You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.\n\nObjective:\n- Answer the user\'s question by verifying whether it is supported by the visual evidence in the medical image and the provided Step of thinking (CoT).\n\nInputs:\n- Question: Does the picture contain liver?\n- Step of thinking (CoT): **Step 1: Imaging Modality**\n\nThe provided image appears to be a computed tomography (CT) scan. This is evident from the cross-sectional view of the body, the use of grayscale to represent different tissue densities, and the presence of a clear outline of the body\'s internal structures. The image\'s high-resolution and detailed representation of the internal organs are also characteristic of CT scans.\n\n**Step 2: Global Im

## Per-model steps (each cell = one step of the original notebook)

In [20]:
# ---------------------------------------------------------------------------
# 1. Load model
# ---------------------------------------------------------------------------
# def load_model(repo_id):
#     extra = {}
#     if "Qwen3-VL" in repo_id:
#         extra["device_map"] = {"": 0}   # Qwen3-VL vision encoder breaks when split across 2 GPUs
#     model, tokenizer = FastVisionModel.from_pretrained(
#         repo_id,
#         load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
#         use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
#         token = HF_TOKEN,
#         **extra,
#     )
#     return model, tokenizer
# ---------------------------------------------------------------------------
# 1. Load model
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 1. Load model
# ---------------------------------------------------------------------------
def load_model(repo_id):
    extra = {}
    if "Qwen3-VL" in repo_id:
        extra["device_map"] = {"": 0}   # Qwen3-VL vision encoder breaks when split across 2 GPUs
    model, tokenizer = FastVisionModel.from_pretrained(
        repo_id,
        load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
        token = HF_TOKEN,
        **extra,
    )
    if "Qwen3-VL" in repo_id:
        # Single T4 only (see above) -> cap image tokens. Qwen3-VL = 32 px per token, so 512*32*32 ≈ 512 tokens max.
        MAX_PX, MIN_PX = 512*32*32, 128*32*32
        ip = tokenizer.image_processor
        import transformers
        from packaging.version import Version
        if Version(transformers.__version__) >= Version("5.0.0"):
            # transformers 5.x: min/max_pixels are read-only properties computed from `size` (a SizeDict)
            from transformers.image_utils import SizeDict
            ip.size = SizeDict(shortest_edge=MIN_PX, longest_edge=MAX_PX)
        else:
            # transformers 4.x: plain attributes
            ip.min_pixels, ip.max_pixels = MIN_PX, MAX_PX
            ip.size = {"shortest_edge": MIN_PX, "longest_edge": MAX_PX}
        print("Qwen3-VL image cap ->", ip.size, "| max_pixels =", ip.max_pixels)
    return model, tokenizer

In [21]:
# # ---------------------------------------------------------------------------
# # 2. Attach LoRA adapters
# # ---------------------------------------------------------------------------
# def attach_lora(model, model_name):
#     model = FastVisionModel.get_peft_model(
#         model,
#         finetune_vision_layers=(model_name != "MedGemma-4B"),   # T4 has no bf16: keep MedGemma's vision tower frozen
#         finetune_language_layers=True,
#         finetune_attention_modules=True,
#         finetune_mlp_modules=True,
#         r=16,
#         lora_alpha=16,
#         lora_dropout=0,
#         bias="none",
#         random_state=3407,
#         use_rslora=False,
#         loftq_config=None,
#     )
#     if model_name == "MedGemma-4B":
#         # T4 trains Gemma3 in float32; upcast any bf16 weights left in the vision tower so layer_norm dtypes match
#         for p in model.parameters():
#             if p.dtype == torch.bfloat16:
#                 p.data = p.data.float()
#         for b in model.buffers():
#             if b.dtype == torch.bfloat16:
#                 b.data = b.data.float()
#     return model
    # ---------------------------------------------------------------------------
# 2. Attach LoRA adapters
# ---------------------------------------------------------------------------
FROZEN_VISION = ("MedGemma-4B", "Qwen3-VL-8B")   # MedGemma: no bf16 on T4. Qwen3-VL: single-GPU memory.

def attach_lora(model, model_name):
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=(model_name not in FROZEN_VISION),
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    if model_name == "MedGemma-4B":
        # T4 trains Gemma3 in float32; upcast any bf16 weights left in the vision tower so layer_norm dtypes match
        for p in model.parameters():
            if p.dtype == torch.bfloat16:
                p.data = p.data.float()
        for b in model.buffers():
            if b.dtype == torch.bfloat16:
                b.data = b.data.float()
    return model

In [22]:
# ---------------------------------------------------------------------------
# 5. Quick check BEFORE fine-tuning (same as the Unsloth notebook does --
#    run one inference with the base model to see what it currently outputs)
# ---------------------------------------------------------------------------
def check_before(model, tokenizer):
    FastVisionModel.for_inference(model)
    sample = cot_df.iloc[0]
    test_instruction = systemPrompt(sample["question"], sample["CoT"])
    test_image = Image.open(resolve_image_path(sample["split_dir"], sample[IMG_COL])).convert("RGB")

    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": test_instruction}
    ]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    output_ids = model.generate(**inputs, max_new_tokens=256, use_cache=True, temperature=1.5, min_p=0.1)
    print("\nBEFORE fine-tuning, model output:")
    print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("Ground truth:", sample["answer"])
    return inputs, sample

In [23]:
# ---------------------------------------------------------------------------
# 6. Train with SFTTrainer + UnslothVisionDataCollator
# ---------------------------------------------------------------------------
# from trl import SFTTrainer, SFTConfig
# from unsloth import is_bf16_supported
# from unsloth.trainer import UnslothVisionDataCollator

# def build_trainer(model, tokenizer):
#     FastVisionModel.for_training(model)

#     trainer = SFTTrainer(
#         model=model,
#         tokenizer=tokenizer,
#         data_collator=UnslothVisionDataCollator(model, tokenizer),
#         train_dataset=converted_dataset,
#         args=SFTConfig(
#             per_device_train_batch_size=2,   # CoT targets are long -- keep batch size low
#             gradient_accumulation_steps=4,
#             warmup_steps=5,
#             #max_steps=30,                  # quick test run -- comment out for full training
#             num_train_epochs=1,
#             learning_rate=2e-4,
#             fp16=not is_bf16_supported(),
#             bf16=is_bf16_supported(),
#             logging_steps=1,
#             optim="adamw_8bit",
#             weight_decay=0.01,
#             lr_scheduler_type="linear",
#             seed=3407,
#             output_dir="outputs",
#             report_to="none",

#             remove_unused_columns=False,
#             dataset_text_field="",
#             dataset_kwargs={"skip_prepare_dataset": True},
#             dataset_num_proc=4,
#             max_length=4096,   # CoT outputs are long (9 sections) -- increased from 2048
#         ),
#     )
#     return trainer
# ---------------------------------------------------------------------------
# 6. Train with SFTTrainer + UnslothVisionDataCollator
# ---------------------------------------------------------------------------
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator

# Per-model overrides. Effective batch (bs x accum) stays 8 everywhere.
TRAIN_OVERRIDES = {
    "Qwen3-VL-8B": dict(per_device_train_batch_size=1, gradient_accumulation_steps=8, max_length=2048),
}

def build_trainer(model, tokenizer, model_name):
    FastVisionModel.for_training(model)

    cfg = dict(per_device_train_batch_size=2,   # CoT prompts are long -- keep batch size low
               gradient_accumulation_steps=4,
               max_length=4096)                 # CoT prompts are long (9 sections)
    cfg.update(TRAIN_OVERRIDES.get(model_name, {}))
    print(f"Train config for {model_name}: {cfg}")

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=converted_dataset,
        args=SFTConfig(
            **cfg,
            warmup_steps=5,
            #max_steps=30,                  # quick test run -- comment out for full training
            num_train_epochs=1,
            learning_rate=2e-4,
            fp16=not is_bf16_supported(),
            bf16=is_bf16_supported(),
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",

            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            dataset_num_proc=4,
        ),
    )
    return trainer

In [24]:
# ---------------------------------------------------------------------------
# 7. Check AFTER fine-tuning -- same sample as before, compare outputs
# ---------------------------------------------------------------------------
def check_after(model, tokenizer, inputs, sample):
    FastVisionModel.for_inference(model)
    output_ids = model.generate(**inputs, max_new_tokens=1024, use_cache=True, temperature=0.3, min_p=0.1)
    print("\nAFTER fine-tuning, model output:")
    print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("\n(Ground truth answer it was trained on:)")
    print(sample["answer"])

In [25]:
# ---------------------------------------------------------------------------
# 8. Save the LoRA adapter locally
# ---------------------------------------------------------------------------
def save_local(model, tokenizer, local_dir):
    model.save_pretrained(local_dir)
    tokenizer.save_pretrained(local_dir)
    print(f"\nSaved locally to ./{local_dir}")

In [26]:
# ---------------------------------------------------------------------------
# 9. Push to Hugging Face Hub
# ---------------------------------------------------------------------------
def push_to_hf(model, tokenizer, hub_repo):
    model.push_to_hub(hub_repo, token=HF_TOKEN)
    tokenizer.push_to_hub(hub_repo, token=HF_TOKEN)
    print(f"Pushed -> https://huggingface.co/{hub_repo}")

In [27]:
# ---------------------------------------------------------------------------
# 10. Free GPU memory + HF download cache before the next model
# ---------------------------------------------------------------------------
def free_gpu():
    for v in ["trainer", "model", "tokenizer", "inputs"]:
        if v in globals(): del globals()[v]
    gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)   # keeps Kaggle disk free
    print("GPU free after cleanup:", [f"{torch.cuda.mem_get_info(i)[0]/1e9:.1f} GB" for i in range(torch.cuda.device_count())])

## Run all models (one after another; each pushed to the Hub before the next starts)

In [28]:
# from huggingface_hub import HfApi
# api = HfApi(token=HF_TOKEN)
# results = {}

# for model_name, repo_id in REPO_IDS.items():
#     HUB_REPO  = f"{HF_USERNAME}/{DATASET_TAG}_closed_withCoT_{model_name}_lora"
#     LOCAL_DIR = f"{DATASET_TAG}_closed_withCoT_{model_name}_lora"

#     if api.repo_exists(HUB_REPO):          # already trained + pushed -> skip (safe re-run after a session ends)
#         print(f"\n[SKIP] {model_name} already on Hub: https://huggingface.co/{HUB_REPO}")
#         results[model_name] = "skipped"
#         continue

#     print("\n" + "="*70)
#     print(f"MODEL: {model_name}  ({repo_id})  ->  {HUB_REPO}")
#     print("="*70)

#     try:
#         model, tokenizer = load_model(repo_id)                 # 1
#         model            = attach_lora(model, model_name)      # 2
#         inputs, sample   = check_before(model, tokenizer)      # 5
#         trainer          = build_trainer(model, tokenizer)     # 6
#         trainer_stats    = trainer.train()
#         check_after(model, tokenizer, inputs, sample)          # 7
#         save_local(model, tokenizer, LOCAL_DIR)                # 8
#         push_to_hf(model, tokenizer, HUB_REPO)                 # 9
#         results[model_name] = "OK"
#     except Exception as e:
#         import traceback; traceback.print_exc()
#         results[model_name] = f"FAILED: {e}"

#     free_gpu()                                                 # 10

# print("\n================ SUMMARY ================")
# for k, v in results.items():
#     print(f"{k:15s} {v}")
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
results = {}

for model_name, repo_id in REPO_IDS.items():
    HUB_REPO  = f"{HF_USERNAME}/{DATASET_TAG}_closed_withCoT_{model_name}_lora"
    LOCAL_DIR = f"{DATASET_TAG}_closed_withCoT_{model_name}_lora"

    if api.repo_exists(HUB_REPO):          # already trained + pushed -> skip (safe re-run after a session ends)
        print(f"\n[SKIP] {model_name} already on Hub: https://huggingface.co/{HUB_REPO}")
        results[model_name] = "skipped"
        continue

    print("\n" + "="*70)
    print(f"MODEL: {model_name}  ({repo_id})  ->  {HUB_REPO}")
    print("="*70)

    try:
        model, tokenizer = load_model(repo_id)                       # 1
        model            = attach_lora(model, model_name)            # 2
        inputs, sample   = check_before(model, tokenizer)            # 5
        trainer          = build_trainer(model, tokenizer, model_name)  # 6
        trainer_stats    = trainer.train()
        check_after(model, tokenizer, inputs, sample)                # 7
        save_local(model, tokenizer, LOCAL_DIR)                      # 8
        push_to_hf(model, tokenizer, HUB_REPO)                       # 9
        results[model_name] = "OK"
    except Exception as e:
        import traceback; traceback.print_exc()
        results[model_name] = f"FAILED: {e}"

    free_gpu()                                                       # 10

print("\n================ SUMMARY ================")
for k, v in results.items():
    print(f"{k:15s} {v}")


[SKIP] MedGemma-4B already on Hub: https://huggingface.co/Arup330/Abdomen_closed_withCoT_MedGemma-4B_lora

[SKIP] Qwen2.5-VL-7B already on Hub: https://huggingface.co/Arup330/Abdomen_closed_withCoT_Qwen2.5-VL-7B_lora

[SKIP] Llama-11B already on Hub: https://huggingface.co/Arup330/Abdomen_closed_withCoT_Llama-11B_lora

MODEL: Qwen3-VL-8B  (Qwen/Qwen3-VL-8B-Instruct)  ->  Arup330/Abdomen_closed_withCoT_Qwen3-VL-8B_lora
==((====))==  Unsloth 2026.9.2: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Unsloth: Offloading embeddings to RAM to save 1.16 GB.
Qwen3-VL image cap -> SizeDict(height=None, width=None, longest_edge=524288, shortest_edge=131072, max_height=None, max_width=None) | max_pixels = 524288


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



BEFORE fine-tuning, model output:
No
Ground truth: No
Train config for Qwen3-VL-8B: {'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 8, 'max_length': 2048}
Unsloth: Model does not have a default image size - using 512


...

Saved model to https://huggingface.co/Arup330/Abdomen_closed_withCoT_Qwen3-VL-8B_lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp2124oj82/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed -> https://huggingface.co/Arup330/Abdomen_closed_withCoT_Qwen3-VL-8B_lora
GPU free after cleanup: ['12.8 GB', '15.5 GB']

[SKIP] Lingshu-7B already on Hub: https://huggingface.co/Arup330/Abdomen_closed_withCoT_Lingshu-7B_lora

================ SUMMARY ================
MedGemma-4B     skipped
Qwen2.5-VL-7B   skipped
Llama-11B       skipped
Qwen3-VL-8B     OK
Lingshu-7B      skipped
